# 06. 評価・コスト・後片付け

**対応するテキスト**: [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md)

> ## ⚠⚠ このノートブックは **必ず最後まで実行してください**
>
> **5 節の後片付けを実施しないと、ハンズオン後も課金が続きます。**
> また **4 節で成果物をダウンロードする前にリソースを削除すると、復旧できません。**

実行順序:

1. 全 Run を 1 枚の表に集約する
2. 技術評価の材料を作る
3. コストを確認する
4. **成果物をダウンロードする**
5. **⚠ 後片付け（必須）**

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
EXPERIMENTS = ["rl-setup-check", "rl-baseline", "rl-reward-exp", "rl-hparam-sweep"]

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

## 1. 全 Run を 1 枚の表に集約する

**これが成果物「実験結果比較表」になります。**

> ⚠ `search_runs` が返すメトリックは **各メトリックの最後の値**です。
> 学習曲線が必要な場合は `MlflowClient().get_metric_history(run_id, key)` を使ってください。
>
> 出典: [Query & compare experiments and runs with MLflow - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow?view=azureml-api-2)

In [ ]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()

existing = []
for name in EXPERIMENTS:
    if mlflow.get_experiment_by_name(name) is not None:
        existing.append(name)
    else:
        print(f"[INFO] 実験 '{name}' は見つかりませんでした（未実行かもしれません）")

if not existing:
    raise RuntimeError(
        "実験が 1 つも見つかりません。notebooks/01・03 を先に実行してください。"
    )

runs = mlflow.search_runs(experiment_names=existing, output_format="pandas")
print(f"\n取得した Run 数: {len(runs)}")
print("列の例:", [c for c in runs.columns if c.startswith("metrics.final")][:5])

In [ ]:
COLS = {
    "tags.mlflow.runName": "run_name",
    "params.env_id": "env_id",
    "params.algo": "algo",
    "params.her_effective": "her",
    "params.reward_mode": "reward_mode",
    "params.shaping_weight": "shaping_w",
    "params.eval_reward_mode": "eval_reward_mode",
    "params.reward_fn_version": "reward_fn_ver",
    "params.learning_rate": "lr",
    "params.gamma": "gamma",
    "params.batch_size": "batch",
    "params.total_timesteps": "timesteps",
    "params.seed": "seed",
    "params.sb3_version": "sb3",
    "params.panda_gym_version": "panda_gym",
    "metrics.final_success_rate": "success_rate",
    "metrics.final_success_rate_stderr": "stderr",
    "metrics.final_mean_reward": "mean_reward",
    "metrics.final_std_reward": "std_reward",
    "metrics.final_mean_episode_length": "mean_len",
    "metrics.train_minutes": "train_min",
    "metrics.video_recorded": "video",
    "run_id": "run_id",
}

if len(runs) == 0:
    print("[ERROR] Run が 1 件も取得できませんでした。")
    print("        notebooks/01・03 が完了しているか、実験名が正しいか確認してください。")
    summary = pd.DataFrame()
else:
    available = {k: v for k, v in COLS.items() if k in runs.columns}
    summary = runs[list(available)].rename(columns=available)

    # success_rate が無い場合（学習ジョブが 1 本も完了していない等）に落ちないようにする
    if "success_rate" in summary.columns:
        summary = summary.sort_values("success_rate", ascending=False, na_position="last")
    else:
        print("[WARN] final_success_rate が記録された Run がありません。並べ替えを省略します。")

    summary.to_csv("all_runs_comparison.csv", index=False, encoding="utf-8-sig")
    print("all_runs_comparison.csv に保存しました")

summary.head(30)

## 2. 技術評価の材料

[docs/09](../docs/09_評価・コスト・後片付け.md) の 9.2「技術評価シート」に貼り付ける数値を作ります。

In [ ]:
if "train_min" in summary.columns:
    total_min = summary["train_min"].fillna(0).sum()
    print(f"実行した Run 数        : {len(summary)}")
    print(f"合計学習時間（分）      : {total_min:.1f}")
    print(f"合計学習時間（時間）    : {total_min / 60:.2f}")

if "success_rate" in summary.columns:
    print(f"\n最高成功率              : {summary['success_rate'].max():.3f}")
    best = summary.loc[summary["success_rate"].idxmax()]
    print(f"  その Run              : {best.get('run_name')}")
    print(f"  条件                  : reward_mode={best.get('reward_mode')}, "
          f"lr={best.get('lr')}, gamma={best.get('gamma')}, seed={best.get('seed')}")

# 再現性: 同一設定でのシード間ばらつき
group_cols = [c for c in ["env_id", "reward_mode", "lr", "gamma", "her"] if c in summary.columns]
if group_cols and "success_rate" in summary.columns:
    rep = (summary.dropna(subset=["success_rate"])
                  .groupby(group_cols)["success_rate"]
                  .agg(["count", "mean", "std"])
                  .query("count >= 2"))
    print("\n=== 再現性（同一設定 × 複数シード） ===")
    print(rep if len(rep) else "（複数シードの Run がまだありません。notebooks/05 の Sweep A を実行してください）")

## 3. コストを確認する

> **本テキストは具体的な金額を記載しません。** 価格はリージョン・VM サイズ・時期で変わるためです。
> **必ず以下で実際の値を確認してください。**

| 確認するもの | 場所 |
|---|---|
| **実際にかかった費用** | Azure Portal → 「コスト管理」→［コスト分析］→ **タグ `project = rl-workshop` でフィルター** |
| VM の時間単価 | [Azure Machine Learning 価格](https://azure.microsoft.com/pricing/details/machine-learning/) |
| 見積もり | [Azure 料金計算ツール](https://azure.microsoft.com/pricing/calculator/) |

下のセルで、コスト確認表に書く「使用量」の情報を集めます。

In [ ]:
cluster = ml_client.compute.get(COMPUTE_NAME)
print("=== コンピューティング クラスター ===")
print(f"  名前                     : {cluster.name}")
print(f"  VM サイズ                : {cluster.size}")
print(f"  min_instances            : {cluster.min_instances}  ← 0 であること")
print(f"  max_instances            : {cluster.max_instances}")
print(f"  idle_time_before_scale_down: {cluster.idle_time_before_scale_down} 秒")

print("\n=== すべてのコンピューティング（種別と状態） ===")
for c in ml_client.compute.list():
    kind = str(getattr(c, "type", "")).lower()
    state = str(getattr(c, "state", "")) or "-"
    is_instance = "instance" in kind or type(c).__name__ == "ComputeInstance"
    mark = "  ⚠ 停止してください" if (is_instance and state.lower() == "running") else ""
    print(f"  {c.name:28s} type={kind:18s} state={state}{mark}")

## 4. 【削除前に必須】成果物をダウンロードする

> ⚠⚠ **リソースを削除すると、MLflow に保存された成果物は復旧できません。**
> **必ず先にダウンロードしてください。**

In [ ]:
import os

WANTED = ["eval_video.mp4", "run_summary.json", "pip_freeze.txt"]
OUT = "workshop_artifacts"
os.makedirs(OUT, exist_ok=True)

if len(summary) == 0:
    print("[ERROR] 集約結果が空です。1 節を先に実行してください。")
else:
    downloaded, skipped = 0, 0
    for _, row in summary.iterrows():
        run_id = row.get("run_id")
        label = str(row.get("run_name") or run_id)
        if not isinstance(run_id, str):
            continue
        try:
            names = {a.path for a in client.list_artifacts(run_id)}
        except Exception as exc:
            print(f"[WARN] {label}: 成果物一覧を取得できません ({exc})")
            continue
        dest = os.path.join(OUT, label)
        for wanted in WANTED:
            if wanted in names:
                client.download_artifacts(run_id=run_id, path=wanted, dst_path=dest)
                downloaded += 1
            else:
                skipped += 1

    print(f"\nダウンロード: {downloaded} 件 / 見つからず: {skipped} 件")
    print(f"保存先: {os.path.abspath(OUT)}")
    print("※ all_runs_comparison.csv も忘れずに手元に保存してください。")

## 5. ⚠ 後片付け（必須）

### 5-1. 実行中のジョブを確認してキャンセルする

> **`sleep infinity` を使った対話型ジョブが残っていると、課金が続きます。**

In [ ]:
ACTIVE = ("Running", "Queued", "Preparing", "Starting", "NotStarted")

active_jobs = []
for job in ml_client.jobs.list():
    if str(job.status) in ACTIVE:
        active_jobs.append(job)
        print(f"⚠ 実行中: {job.name}  status={job.status}  name={job.display_name}")

if not active_jobs:
    print("実行中のジョブはありません。")

In [ ]:
# ⚠ 実行すると上で列挙されたジョブをすべてキャンセルします。
#    必要なジョブが走っていないか確認してから実行してください。
#
# for job in active_jobs:
#     ml_client.jobs.begin_cancel(job.name)
#     print("キャンセル要求:", job.name)

### 5-2. コンピューティング インスタンスを停止する

> **停止すれば課金は止まります**（ただしクォータは解放されません）。
> 完全に不要なら削除してください。

In [ ]:
# ⚠ 自分のインスタンス名に置き換えてください
# COMPUTE_INSTANCE_NAME = "ci-<yourname>"
#
# ml_client.compute.begin_stop(COMPUTE_INSTANCE_NAME).wait()
# print("停止しました:", COMPUTE_INSTANCE_NAME)
#
# 完全に削除する場合:
# ml_client.compute.begin_delete(COMPUTE_INSTANCE_NAME).wait()

print("上のコメントを外して実行するか、Azure ML studio の［コンピューティング］から停止してください。")

### 5-3. すべて不要な場合 — リソース グループごと削除する

> ⚠⚠ **元に戻せません。4 節のダウンロードを完了してから実行してください。**

```powershell
az group delete --name rg-rl-workshop-<yourname> --yes --no-wait
```

### 5-4. 残す場合の判断

| 判断 | 残すもの | 消すもの |
|---|---|---|
| 継続検証する | ワークスペース、環境、MLflow の記録 | **コンピューティング インスタンス**（停止で可） |
| 記録だけ残す | ワークスペース（記録の保管庫） | コンピューティング一式 |
| すべて終了 | ダウンロードした成果物のみ | **リソース グループごと** |

> **重要**: ワークスペースを残す場合も、**ストレージ アカウントと Container Registry には少額の課金が継続します。**
> Microsoft Cost Management で定期的に確認してください。

## 6. ✅ 最終チェックリスト

- [ ] `all_runs_comparison.csv` を作成し、**手元に保存した**
- [ ] 技術評価シートに貼る数値を集めた（2 節）
- [ ] **Microsoft Cost Management で実際の費用を確認した**（3 節）
- [ ] **`workshop_artifacts/` に成果物をダウンロードした**（4 節）
- [ ] **実行中のジョブが残っていないことを確認した**（5-1）
- [ ] **コンピューティング インスタンスを停止した**（5-2）
- [ ] リソースの継続・削除方針を決めた（5-4）

---

**お疲れさまでした。**

技術評価シートと次ステップ案の書き方は [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md) の 9.2 / 9.7 を参照してください。